## Description
This script processes the compiled CAR data (from Krish) to create a clean, session-level dataset for Power BI analysis. It groups the data by "Contact Session ID" and aggregates key metrics such as call start time, starting hour, unique activity count, and total call duration in minutes. It also adds the call’s weekday and date for time-based analysis.The resulting CSV ("combined_calls_transformed_callduration_fixed.csv") is optimized for Power BI visuals, enabling metrics like average call duration or activity patterns by hour and day.

Note that, in a previous version of this code, Call Duration was taken from the difference between the first and last timestamp of a particular Contact Session ID. Since this inflated the average call durations with callbacks that occured hours (or even days) after the start of the call, I changed the way it is calculated here. Now, calls which contain callbacks do not include the time between the original call and any further callbacks. For more information on how/why I did this, refer to my documentation for Presentation 3.

In [3]:
import pandas as pd

# --- Load and prepare data ---
df = pd.read_csv("combined_calls.csv")
df['Activity Start Timestamp'] = pd.to_datetime(df['Activity Start Timestamp'], errors='coerce')
df = df.sort_values(['Contact Session ID', 'Activity Start Timestamp'])

# --- Function to compute adjusted call duration per session ---
def calculate_adjusted_duration(group):
    group = group.sort_values('Activity Start Timestamp')
    activities = group['Activity Name'].tolist()
    times = group['Activity Start Timestamp'].tolist()

    # Always ignore the very last timestamp (wrap-up)
    if len(times) > 1:
        times = times[:-1]
        activities = activities[:-1]

    # Case 1: No DisconnectContact1 or DisconnectContact → full duration (minus wrap-up)
    if "DisconnectContact1" not in activities and "DisconnectContact" not in activities:
        return (times[-1] - times[0]).total_seconds() / 60

    # Case 2: Outbound call — starts *after* DisconnectContact and ends *before* final timestamp
    if "DisconnectContact" in activities and "DisconnectContact1" not in activities:
        idx = activities.index("DisconnectContact")
        if idx + 1 < len(times) - 1:
            start_time = times[idx + 1]
            end_time = times[-1]  # (already trimmed wrap-up above)
            return (end_time - start_time).total_seconds() / 60
        else:
            return 0

    # Case 3: Has DisconnectContact1 but no CallbackRetry
    if "DisconnectContact1" in activities and "CallbackRetry" not in activities:
        disconnect_time = times[activities.index("DisconnectContact1")]
        # Find LegalServerScreenPop *after* the disconnect
        after_disconnect = [t for a, t in zip(activities, times)
                            if a == "LegalServerScreenPop" and t > disconnect_time]
        if after_disconnect:
            resume_time = after_disconnect[0]
            return ((disconnect_time - times[0]) + (times[-1] - resume_time)).total_seconds() / 60
        else:
            # No resume → ends at disconnect
            return (disconnect_time - times[0]).total_seconds() / 60

    # Case 4: Has both DisconnectContact1 and CallbackRetry
    total_duration = 0
    active = True
    start_time = times[0]

    for i, act in enumerate(activities):
        if act == "DisconnectContact1" and active:
            total_duration += (times[i] - start_time).total_seconds() / 60
            active = False
        elif act == "LegalServerScreenPop" and not active:
            start_time = times[i]
            active = True
        elif act == "CallbackRetry" and active:
            # Pause before CallbackRetry (include up to previous event)
            if i > 0:
                total_duration += (times[i-1] - start_time).total_seconds() / 60
            active = False
        elif act == "LegalServerScreenPop" and not active:
            start_time = times[i]
            active = True

    # If still active at the end, close the last segment
    if active:
        total_duration += (times[-1] - start_time).total_seconds() / 60

    return total_duration

# --- Apply per Contact Session ID ---
durations = (
    df.groupby('Contact Session ID', group_keys=False)
      .apply(calculate_adjusted_duration)
      .reset_index(name='Adjusted_Call_Duration_Minutes')
)

# --- Save results ---
durations.to_csv("adjusted_call_durations.csv", index=False)

print(durations.head())


C:\Users\julia\AppData\Local\Temp\ipykernel_15124\1193706197.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("combined_calls.csv")


                     Contact Session ID  Adjusted_Call_Duration_Minutes
0  00002422-f51f-458b-82d6-cfa5a3f36fd9                        0.333333
1  0000a8d5-cecb-46b1-82cf-b7ce07d85b24                        0.750000
2  00011655-35de-476f-9a8c-dd48ed4d914a                        3.316667
3  00014a58-a6ce-4cb2-a529-d55e2c9c304d                        2.066667
4  00015327-f646-462f-a585-0552331eed4e                        0.200000


In [4]:
import pandas as pd

# --- Step 1: Load main dataset ---
df = pd.read_csv("combined_calls.csv")
df['Activity Start Timestamp'] = pd.to_datetime(df['Activity Start Timestamp'], errors='coerce')

# --- Step 2: Apply adjusted duration calculation ---
durations = (
    df.groupby('Contact Session ID', group_keys=False)
      .apply(calculate_adjusted_duration)
      .reset_index(name='Adjusted_Call_Duration_Minutes')
)

# --- Step 3: Aggregate data for Power BI ---
grouped = (
    df.groupby('Contact Session ID')
    .agg(
        Call_Start_Time=('Activity Start Timestamp', 'min'),
        Starting_Hour=('Activity Start Timestamp', lambda x: x.min().hour),
        Count=('Activity Start Timestamp', lambda x: len(set(x))),
        )
    .reset_index()
)

grouped['DayOfWeekNum'] = grouped['Call_Start_Time'].dt.dayofweek + 1
grouped['Call_Start_Date'] = grouped['Call_Start_Time'].dt.date

# --- Step 4: Merge with the adjusted durations ---
merged = pd.merge(grouped, durations, on='Contact Session ID', how='left')

# --- Step 5: Save for Power BI ---
merged.to_csv("combined_calls_transformed_callduration_fixed.csv", index=False)

print(merged.head())


C:\Users\julia\AppData\Local\Temp\ipykernel_15124\2582705413.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("combined_calls.csv")


                     Contact Session ID     Call_Start_Time  Starting_Hour  \
0  00002422-f51f-458b-82d6-cfa5a3f36fd9 2025-03-13 12:51:21             12   
1  0000a8d5-cecb-46b1-82cf-b7ce07d85b24 2025-03-17 16:53:15             16   
2  00011655-35de-476f-9a8c-dd48ed4d914a 2024-11-06 14:39:01             14   
3  00014a58-a6ce-4cb2-a529-d55e2c9c304d 2025-02-28 08:33:40              8   
4  00015327-f646-462f-a585-0552331eed4e 2025-06-03 07:43:56              7   

   Count  DayOfWeekNum Call_Start_Date  Adjusted_Call_Duration_Minutes  
0      4             4      2025-03-13                        0.333333  
1      5             1      2025-03-17                        0.750000  
2     10             3      2024-11-06                        3.316667  
3      8             5      2025-02-28                        2.066667  
4      3             2      2025-06-03                        0.200000  
